In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Point
import datetime
from scipy.spatial.transform import Rotation as R
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
import os
import warnings

# Suppress pandas chained assignment warnings and local CRS warnings
warnings.filterwarnings('ignore')
pd.options.mode.chained_assignment = None

# Force Plotly to render inline in the Jupyter Notebook by default
pio.renderers.default = "notebook"

In [ ]:
input_path = r"C:\streaming_emulator\Driver behavior\Data\output\raw_data_combined.csv"
df = pd.read_csv(input_path)

df['Time'] = pd.to_datetime(df['Time'], utc=True)
df = df.sort_values(by=['Trip Number', 'Time']).reset_index(drop=True)

# 1. Dynamic Delta T
df['Delta_t_sec'] = df.groupby('Trip Number')['Time'].diff().dt.total_seconds().fillna(0.0)

# 2. Gap Severance (If signal lost for > 30s, create a new segment to isolate physics)
df['is_gap'] = df['Delta_t_sec'] > 30.0
df['Segment_ID'] = df.groupby('Trip Number')['is_gap'].cumsum()

# 3. Exact Distance Integration
df['Distance_m'] = (df['Speed (GPS)(km/h)'] * (5.0 / 18.0)) * df['Delta_t_sec']

In [ ]:
# We MUST calibrate before trimming, to ensure we catch the driveway idling.
df['Acc_X_cal'] = df['Acceleration Sensor(X axis)(g)']
df['Acc_Y_cal'] = df['Acceleration Sensor(Y axis)(g)']
df['Acc_Z_cal'] = df['Acceleration Sensor(Z axis)(g)']

for trip_id, trip_data in df.groupby('Trip Number'):
    # Find segments where speed is 0
    stationary = trip_data[trip_data['Speed (GPS)(km/h)'] == 0]
    
    if not stationary.empty:
        med_x = stationary['Acceleration Sensor(X axis)(g)'].median()
        med_y = stationary['Acceleration Sensor(Y axis)(g)'].median()
        med_z = stationary['Acceleration Sensor(Z axis)(g)'].median()
    else:
        med_x, med_y, med_z = 0.0, 0.0, 1.0 
        
    pitch = np.arctan2(-med_x, np.sqrt(med_y**2 + med_z**2))
    roll = np.arctan2(med_y, np.sqrt(med_x**2 + med_z**2))
    rot = R.from_euler('xy', [-roll, -pitch], degrees=False)
    
    acc_vectors = trip_data[['Acceleration Sensor(X axis)(g)', 'Acceleration Sensor(Y axis)(g)', 'Acceleration Sensor(Z axis)(g)']].values
    calibrated_vectors = rot.apply(acc_vectors)
    
    idx = trip_data.index
    df.loc[idx, 'Acc_X_cal'] = calibrated_vectors[:, 0]
    df.loc[idx, 'Acc_Y_cal'] = calibrated_vectors[:, 1]
    df.loc[idx, 'Acc_Z_cal'] = calibrated_vectors[:, 2]

In [ ]:
# 1. The Privacy Trim
df['Cum_Dist_m'] = df.groupby('Trip Number')['Distance_m'].cumsum()
df['Rev_Cum_Dist_m'] = df.iloc[::-1].groupby('Trip Number')['Distance_m'].cumsum().iloc[::-1]
df = df[(df['Cum_Dist_m'] > 1000.0) & (df['Rev_Cum_Dist_m'] > 1000.0)].reset_index(drop=True)

# 2. Terrain Smoothing & Road Grade (Assuming column 'Altitude(m)' exists)
if 'Altitude(m)' in df.columns:
    # 10-row rolling median (~40 sec window) to crush GPS multipath noise
    df['Smoothed_Alt'] = df.groupby('Trip Number')['Altitude(m)'].transform(lambda x: x.rolling(10, min_periods=1, center=True).median())
    df['Delta_Alt'] = df.groupby(['Trip Number', 'Segment_ID'])['Smoothed_Alt'].diff().fillna(0)
    df['Grade_Rad'] = np.arctan2(df['Delta_Alt'], df['Distance_m'].replace(0, np.nan)).fillna(0)
else:
    df['Grade_Rad'] = 0.0 # Fallback if Altitude is missing

# 3. Steering Geometry (Heading & Yaw Rate)
def calc_bearing(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlon = lon2 - lon1
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - (np.sin(lat1) * np.cos(lat2) * np.cos(dlon))
    return np.degrees(np.arctan2(x, y)) % 360

df['Heading'] = calc_bearing(df['Latitude'].shift(1), df['Longitude'].shift(1), df['Latitude'], df['Longitude']).fillna(0)
df['Delta_Heading'] = df.groupby(['Trip Number', 'Segment_ID'])['Heading'].diff().fillna(0)
df['Delta_Heading'] = (df['Delta_Heading'] + 180) % 360 - 180 # Fix 360 wrap-around
df['Yaw_Rate_deg_s'] = (df['Delta_Heading'] / df['Delta_t_sec'].replace(0, np.nan)).abs().fillna(0)

# 4. Geospatial Context Injection (R-Tree Join)
geometry = [Point(xy) for xy in zip(df['Longitude'], df['Latitude'])]
gdf_telemetry = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")

minx, miny, maxx, maxy = gdf_telemetry.total_bounds
bbox = (minx - 0.01, miny - 0.01, maxx + 0.01, maxy + 0.01)

shape_path = r"C:\streaming_emulator\Driver behavior\Data\shapefiles\North_India\gis_osm_roads_free_1.shp"
print("Loading OSM bounding box & executing R-Tree Join...")
gdf_roads = gpd.read_file(shape_path, bbox=bbox)

# Nearest-neighbor spatial join
gdf_joined = gpd.sjoin_nearest(gdf_telemetry, gdf_roads[['geometry', 'fclass']], how='left', distance_col='dist_to_road')
df = pd.DataFrame(gdf_joined.drop(columns=['geometry']))
df['Road_Type'] = df['fclass'].fillna('unknown')

In [ ]:
# Apply the true physics correction for inclines/declines
df['Acc_X_True'] = df['Acc_X_cal'] + np.sin(df['Grade_Rad'])

# Speed Delta for acceleration check
df['Delta_V'] = df.groupby(['Trip Number', 'Segment_ID'])['Speed (GPS)(km/h)'].diff().fillna(0)

df['Event_Harsh_Braking'] = df['Acc_X_True'] < -0.35
df['Event_Harsh_Accel'] = (df['Acc_X_True'] > 0.25) & (df['Delta_V'] > 0)

# Adaptive Cornering
high_speed_roads = ['motorway', 'motorway_link', 'trunk', 'trunk_link', 'primary']
df['Cornering_Threshold'] = np.where(
    df['Road_Type'].isin(high_speed_roads) & (df['Yaw_Rate_deg_s'] < 5.0),
    0.25,  # Strict threshold for straight highway swerves
    0.45   # Loose threshold for sharp urban turns
)
df['Event_Harsh_Cornering'] = (df['Acc_Y_cal'].abs() > df['Cornering_Threshold']) & (df['Speed (GPS)(km/h)'] > 20.0)

# Export
cols_to_drop = ['is_gap', 'Cum_Dist_m', 'Rev_Cum_Dist_m', 'Delta_Alt', 'Delta_Heading', 'Cornering_Threshold', 'Delta_V']
df.drop(columns=[c for c in cols_to_drop if c in df.columns], inplace=True)

output_dir = r"C:\streaming_emulator\Driver behavior\Data\results"
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, f"results_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.csv")
df.to_csv(output_path, index=False)
print(f"Intelligent Telematics saved to: {output_path}")

In [ ]:
fig_gg = px.scatter(
    df, x='Acc_Y_cal', y='Acc_X_True', color='Speed (GPS)(km/h)',
    title="Grade-Compensated Traction Circle",
    labels={'Acc_Y_cal': 'Lateral G', 'Acc_X_True': 'True Longitudinal G (Grade Adjusted)'},
    color_continuous_scale=px.colors.sequential.Viridis, opacity=0.6
)
fig_gg.add_shape(type="circle", x0=-1, y0=-1, x1=1, y1=1, line_color="red", line_dash="dash")
fig_gg.add_hline(y=0, line_width=1, line_color="black")
fig_gg.add_vline(x=0, line_width=1, line_color="black")
fig_gg.update_yaxes(range=[-1.5, 1.5], scaleanchor="x", scaleratio=1)
fig_gg.update_xaxes(range=[-1.5, 1.5])
fig_gg.show() # Renders inline

In [ ]:
sample_trip = df['Trip Number'].iloc[0]
df_trip = df[df['Trip Number'] == sample_trip]

fig_tel = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.05,
                        subplot_titles=("Speed vs Altitude", "Heading & Yaw Rate", "True G-Forces"))

fig_tel.add_trace(go.Scatter(x=df_trip['Time'], y=df_trip['Speed (GPS)(km/h)'], name='Speed'), row=1, col=1)
if 'Smoothed_Alt' in df_trip.columns:
    fig_tel.add_trace(go.Scatter(x=df_trip['Time'], y=df_trip['Smoothed_Alt'], name='Altitude', yaxis="y2"), row=1, col=1)

fig_tel.add_trace(go.Scatter(x=df_trip['Time'], y=df_trip['Yaw_Rate_deg_s'], name='Yaw Rate', fill='tozeroy'), row=2, col=1)

fig_tel.add_trace(go.Scatter(x=df_trip['Time'], y=df_trip['Acc_X_True'], name='True Lon G'), row=3, col=1)
fig_tel.add_trace(go.Scatter(x=df_trip['Time'], y=df_trip['Acc_Y_cal'], name='Lat G'), row=3, col=1)

fig_tel.update_layout(height=800, hovermode="x unified", title_text=f"Telemetry: Trip {sample_trip}")
fig_tel.show() # Renders inline

In [ ]:
df_plot = df.copy()
conditions = [df_plot['Event_Harsh_Braking'], df_plot['Event_Harsh_Accel'], df_plot['Event_Harsh_Cornering']]
choices = ['Harsh Braking', 'Harsh Accel', 'Harsh Cornering']
df_plot['Event_Type'] = np.select(conditions, choices, default='Normal')

df_plot['is_event'] = df_plot['Event_Type'] != 'Normal'
df_plot = df_plot.sort_values(by='is_event')

fig_map = px.scatter_mapbox(
    df_plot, lat="Latitude", lon="Longitude", color="Event_Type",
    color_discrete_map={'Normal': '#1f77b4', 'Harsh Braking': '#d62728', 'Harsh Accel': '#2ca02c', 'Harsh Cornering': '#ff7f0e'},
    hover_data={"is_event": False, "Road_Type": True, "Grade_Rad": ":.3f", "Yaw_Rate_deg_s": ":.1f", "Acc_X_True": ":.2f", "Acc_Y_cal": ":.2f"},
    zoom=11, title="Context-Aware Driver Behavior Map"
)

fig_map.update_layout(mapbox_style="open-street-map", height=900, margin={"r":0,"t":40,"l":0,"b":0})

for trace in fig_map.data:
    if trace.name == 'Normal':
        trace.marker.size = 3
        trace.marker.opacity = 0.3
    else:
        trace.marker.size = 18
        trace.marker.opacity = 1.0

# This command pushes the map directly to a new tab in your default web browser
fig_map.show(renderer="browser")

In [ ]:
# 1. Macro-Level Event Normalization (Events per 100km)
total_dist_km = df['Distance_m'].sum() / 1000.0

if total_dist_km > 0:
    rate_brake = (df['Event_Harsh_Braking'].sum() / total_dist_km) * 100.0
    rate_accel = (df['Event_Harsh_Accel'].sum() / total_dist_km) * 100.0
    rate_corner = (df['Event_Harsh_Cornering'].sum() / total_dist_km) * 100.0
else:
    rate_brake = rate_accel = rate_corner = 0.0

fig_radar = go.Figure(data=go.Scatterpolar(
  r=[rate_brake, rate_accel, rate_corner, rate_brake],
  theta=['Harsh Braking', 'Harsh Accel', 'Harsh Cornering', 'Harsh Braking'],
  fill='toself',
  line_color='red'
))

fig_radar.update_layout(
  polar=dict(radialaxis=dict(visible=True, title="Events per 100km")),
  title="Driver Risk Profile (Radar)",
  height=500
)

# 2. Contextual Speed Adherence (Violin Plot)
fig_violin = px.violin(
    df, y="Speed (GPS)(km/h)", x="Road_Type", color="Road_Type",
    box=True, points="all", hover_data=["Acc_X_True"],
    title="Speed Distribution by Road Classification"
)

fig_violin.update_layout(height=500, showlegend=False)

fig_radar.show()
fig_violin.show()

In [ ]:
events_brake = df['Event_Harsh_Braking'].values
events_accel = df['Event_Harsh_Accel'].values
events_corner = df['Event_Harsh_Cornering'].values
distances = df['Distance_m'].values

scores = np.zeros(len(df))
current_score = 100.0
clean_distance_accum = 0.0

P_BRAKE = 5.0
P_ACCEL = 3.0
P_CORNER = 5.0
RECOVERY_DIST_THRESHOLD = 250.0 

for i in range(len(df)):
    penalty = 0.0
    if events_brake[i]: penalty += P_BRAKE
    if events_accel[i]: penalty += P_ACCEL
    if events_corner[i]: penalty += P_CORNER
    
    if penalty > 0:
        current_score -= penalty
        clean_distance_accum = 0.0 
    else:
        clean_distance_accum += distances[i]
        if clean_distance_accum >= RECOVERY_DIST_THRESHOLD:
            current_score += 1.0
            clean_distance_accum -= RECOVERY_DIST_THRESHOLD 
            
    if current_score > 100.0: current_score = 100.0
    if current_score < 0.0: current_score = 0.0
        
    scores[i] = current_score

df['RealTime_Score'] = scores

# Overwrite previous output CSV to include the new RealTime_Score column
output_dir = r"C:\streaming_emulator\Driver behavior\Data\results"
existing_files = [f for f in os.listdir(output_dir) if f.startswith("results_")]
if existing_files:
    latest_file = sorted(existing_files)[-1] 
    output_path = os.path.join(output_dir, latest_file)
else:
    output_path = os.path.join(output_dir, f"results_scored_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.csv")

df.to_csv(output_path, index=False)

In [ ]:
sample_trip = df['Trip Number'].iloc[0]
df_trip = df[df['Trip Number'] == sample_trip]

fig_score = go.Figure()

fig_score.add_trace(go.Scatter(
    x=df_trip['Time'], y=df_trip['RealTime_Score'],
    mode='lines', name='Driver Score',
    line=dict(color='blue', width=2)
))

event_mask = df_trip['Event_Harsh_Braking'] | df_trip['Event_Harsh_Accel'] | df_trip['Event_Harsh_Cornering']
df_events = df_trip[event_mask]

fig_score.add_trace(go.Scatter(
    x=df_events['Time'], y=df_events['RealTime_Score'],
    mode='markers', name='Infraction Applied',
    marker=dict(color='red', size=10, symbol='x')
))

fig_score.update_layout(
    title=f"Leaky Bucket Dynamics: Real-Time Score Trace for Trip {sample_trip}",
    yaxis=dict(title="Score (0-100)", range=[0, 105]),
    hovermode="x unified", height=500
)

fig_score.show()

In [ ]:
fig_score_map = px.scatter_mapbox(
    df, 
    lat="Latitude", lon="Longitude", 
    color="RealTime_Score",
    color_continuous_scale="RdYlGn", 
    range_color=[50, 100], 
    hover_name="RealTime_Score",
    hover_data={
        "Latitude": False, "Longitude": False,
        "Speed (GPS)(km/h)": ":.1f",
        "Road_Type": True,
        "Event_Harsh_Braking": True,
        "Event_Harsh_Cornering": True,
        "Event_Harsh_Accel": True
    },
    zoom=12,
    title="Real-Time Score Degradation and Recovery Trace"
)

fig_score_map.update_layout(
    mapbox_style="open-street-map", 
    height=900, 
    margin={"r":0,"t":40,"l":0,"b":0}
)

fig_score_map.update_traces(marker=dict(size=6, opacity=1.0))

fig_score_map.show(renderer="browser")